# Trend + Pullback Trading Strategy Backtest

This notebook implements a simple trend-following system with:
- EMA trend filter
- RSI pullback entries
- ATR-based stop loss and take profit


In [ ]:
import pandas as pd
import numpy as np

# Load your data here (CSV must have: open, high, low, close)
df = pd.read_csv('eth_binance_3y.csv')

# Ensure correct format
df.columns = [col.lower() for col in df.columns]

In [ ]:
def add_indicators(df):
    df['ema50'] = df['close'].ewm(span=50).mean()
    df['ema200'] = df['close'].ewm(span=200).mean()

    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))

    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    df['atr'] = ranges.max(axis=1).rolling(14).mean()

    return df

df = add_indicators(df)

In [ ]:
def generate_signals(df):
    df['signal'] = 0

    buy_condition = (
        (df['close'] > df['ema200']) &
        (df['close'] < df['ema50']) &
        (df['rsi'] < 40)
    )

    sell_condition = (
        (df['close'] < df['ema200']) &
        (df['close'] > df['ema50']) &
        (df['rsi'] > 60)
    )

    df.loc[buy_condition, 'signal'] = 1
    df.loc[sell_condition, 'signal'] = -1

    return df

df = generate_signals(df)

In [ ]:
def backtest(df, initial_balance=10000, risk_per_trade=0.01):
    balance = initial_balance
    position = 0
    entry_price = 0

    for i in range(1, len(df)):
        if position == 0:
            if df['signal'].iloc[i] == 1:
                position = 1
                entry_price = df['close'].iloc[i]
                atr = df['atr'].iloc[i]
                sl = entry_price - atr
                tp = entry_price + 2 * atr

            elif df['signal'].iloc[i] == -1:
                position = -1
                entry_price = df['close'].iloc[i]
                atr = df['atr'].iloc[i]
                sl = entry_price + atr
                tp = entry_price - 2 * atr

        elif position == 1:
            if df['low'].iloc[i] <= sl:
                balance *= (1 - risk_per_trade)
                position = 0
            elif df['high'].iloc[i] >= tp:
                balance *= (1 + 2 * risk_per_trade)
                position = 0

        elif position == -1:
            if df['high'].iloc[i] >= sl:
                balance *= (1 - risk_per_trade)
                position = 0
            elif df['low'].iloc[i] <= tp:
                balance *= (1 + 2 * risk_per_trade)
                position = 0

    return balance

final_balance = backtest(df)
print('Final Balance:', final_balance)